## Data Processing

In [9]:
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.dataset as pa_dataset
import pyarrow.compute as pc
import pandas as pd
import numpy as np

from pathlib import Path
import gc

In [2]:
# file directories

file_path = Path("../data/Global Factor_2000.parquet")
FEATURES_RAW = Path("../data/features_clean.parquet")
TARGET_FILE = Path("../data/target.parquet")
FEATURES_FLAG = Path("../data/features_with_flags.parquet")
FEATURES_FINAL = Path("../data/features_ranked.parquet")

schema = pq.read_schema(file_path)
metadata = pq.read_metadata(file_path)
n_rows = metadata.num_rows
n_columns = len(schema.names)

print(f"Row count:{n_rows}, Column count: {n_columns}")

parquet_file = pq.ParquetFile(file_path)
small_batch = next(parquet_file.iter_batches(batch_size=1000))
sample_df = pa.Table.from_batches([small_batch]).to_pandas()
sample_df.head()

Row count:11625498, Column count: 443


,obs_main,exch_main,common,primary_sec,gvkey,date,permno,permco,iid,id,...,dltnetis_mev,dstnetis_mev,dbnetis_mev,netis_mev,fincf_mev,ivol_capm_60m,beta_21d,beta_252d,rvol_252d,rvolhl_21d
0,1.0,1.0,1.0,1.0,185909,2015-08-31,NaN,NaN,01C,218590901.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.065673
1,1.0,1.0,1.0,1.0,279546,2014-01-29,NaN,NaN,01W,327954601.0,...,NaN,NaN,NaN,NaN,NaN,0.103203,NaN,NaN,NaN,0.001436
2,1.0,1.0,1.0,1.0,279576,2014-01-29,NaN,NaN,01W,327957601.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.001688
3,1.0,1.0,1.0,1.0,040080,2023-05-31,22750.0,59043.0,02,22750.0,...,NaN,NaN,NaN,NaN,NaN,NaN,0.039769,0.011270,0.003444,0.003558
4,1.0,1.0,1.0,1.0,205572,2006-07-31,NaN,NaN,01W,320557201.0,...,NaN,NaN,NaN,NaN,NaN,0.079699,NaN,0.578906,0.033397,0.008792


In [3]:
# parquet_file = pq.ParquetFile(file_path)
# results = []

# for i, batch in enumerate(parquet_file.iter_batches(batch_size=500_000)):

#     batch_table = pa.Table.from_batches([batch])
#     df_batch = batch_table.to_pandas()

#     nan_counts = df_batch.isnull().sum(axis=1)
#     results.append(nan_counts)

#     print(f"Batch {i + 1} processed | rows: {len(df_batch)}")

# nan_per_row = pd.concat(results, ignore_index=True)
# print(f"Total rows: {len(nan_per_row)}")
# print(f"Rows with zero NaNs: {(nan_per_row == 0).sum()}")
# print(f"Rows with at least one NaN: {(nan_per_row > 0).sum()}")
# print(f"Average NaNs per row: {nan_per_row.mean():.2f}")
# print(f"Max NaNs in a single row: {nan_per_row.max()}")

In [4]:
raw_columns = schema.names
raw_n_rows  = metadata.num_rows

COLUMNS_TO_REMOVE: set[str] = {

    # Identifiers and keys
    "permno", "permco", "eom",

    # data-quality flags and others
    "obs_main", "exch_main", "primary_sec", "common", "size_grp",
    "ret_lag_dif", "adjfct", "bidask", "comp_tpci", "crsp_shrcd",
    "comp_exchg", "crsp_exchcd", "source_crsp", "curcd",

    # Target variable (isolated separately)
    "ret_exc_lead1m",

    # returns (direct look-ahead risk)
    "ret_exc", "ret", "ret_local",

    # Raw unscaled accounting numbers (dimensionless variants are retained)
    "enterprise_value", "book_equity", "assets", "sales", "net_income",

    # Exchange rates and industry codes
    "fx", "gics", "naics", "sic", "ff49",

    # Raw price and volume data
    "prc", "prc_local", "prc_high", "prc_low", "shares", "tvol",
}

TARGET_COLUMN = "ret_exc_lead1m"

# Partition the raw columns into feature and target lists. The feature list
# preserves the original parquet ordering for column-projection efficiency
feature_cols = [c for c in raw_columns if c not in COLUMNS_TO_REMOVE and c != TARGET_COLUMN]
target_cols = [TARGET_COLUMN] if TARGET_COLUMN in raw_columns else []

print(f"Columns removed: {len(COLUMNS_TO_REMOVE & set(raw_columns))}")
print(f"Features kept: {len(feature_cols)}")
print(f"Target present: {bool(target_cols)}")

Columns removed: 37
Features kept: 406
Target present: True


In [5]:
# Stream the raw parquet in batches, projecting onto feature and target columns
# and writing to separate output files. Lazy writer creation ensures the output
# schemas are inferred from the actual projected batch.

parquet_file = pq.ParquetFile(file_path)
feature_writer = None
target_writer = None
total_rows = 0

try:
    for i, batch in enumerate(parquet_file.iter_batches(batch_size=500_000)):
        batch_table = pa.Table.from_batches([batch])

        feature_table = batch_table.select(feature_cols)
        if feature_writer is None:
            feature_writer = pq.ParquetWriter(FEATURES_RAW, feature_table.schema, compression="snappy")
        feature_writer.write_table(feature_table)

        if target_cols:
            target_table = batch_table.select(target_cols)
            if target_writer is None:
                target_writer = pq.ParquetWriter(TARGET_FILE, target_table.schema, compression="snappy")
            target_writer.write_table(target_table)

        total_rows += batch_table.num_rows
        print(f"Batch {i + 1:2d}  rows ={batch_table.num_rows:>8,}  cumulative ={total_rows:>10,}")
finally:
    if feature_writer is not None:
        feature_writer.close()
    if target_writer is not None:
        target_writer.close()

Batch  1  rows = 500,000  cumulative =   500,000
Batch  2  rows = 500,000  cumulative = 1,000,000
Batch  3  rows = 500,000  cumulative = 1,500,000
Batch  4  rows = 500,000  cumulative = 2,000,000
Batch  5  rows = 500,000  cumulative = 2,500,000
Batch  6  rows = 500,000  cumulative = 3,000,000
Batch  7  rows = 500,000  cumulative = 3,500,000
Batch  8  rows = 500,000  cumulative = 4,000,000
Batch  9  rows = 500,000  cumulative = 4,500,000
Batch 10  rows = 500,000  cumulative = 5,000,000
Batch 11  rows = 500,000  cumulative = 5,500,000
Batch 12  rows = 500,000  cumulative = 6,000,000
Batch 13  rows = 500,000  cumulative = 6,500,000
Batch 14  rows = 500,000  cumulative = 7,000,000
Batch 15  rows = 500,000  cumulative = 7,500,000
Batch 16  rows = 500,000  cumulative = 8,000,000
Batch 17  rows = 500,000  cumulative = 8,500,000
Batch 18  rows = 500,000  cumulative = 9,000,000
Batch 19  rows = 500,000  cumulative = 9,500,000
Batch 20  rows = 500,000  cumulative =10,000,000
Batch 21  rows = 500

### K0 and K1 groups

In [6]:
# K1 patterns identify slow-moving accounting characteristics that receive
# the annual lag structure at months {12, 24, 36, 48, 60} in section 1.5.
k1_factors: tuple[str, ...] = (
    # Valuation ratios
    "be_me", "bm", "ev_", "sale_me", "div_", "eps_",

    # Profitability
    "gp_", "op_", "ni_", "ope_", "roe_", "roa_", "fcf_", "ebit",

    # Investment and growth
    "at_gr", "inv_gr", "capx_", "noa_gr",

    # Leverage and solvency
    "debt_", "lev_", "ltdebt_",
    
    # Accounting quality
    "accruals_", "tang_",
)

# K0 patterns identify fast-moving characteristics that are kept at the
# current value only. The pattern list is illustrative; final assignment
# defaults to K0 for any retained characteristic not matching the K1 patterns.
k0_factors: tuple[str, ...] = (
    "ret_", "rvol_", "ivol_", "beta_", "skew_",
    "kurt_", "mom_", "rmax_", "rmin_", "dolvol_",
    "turnover_", "amihud_", "iliq_", "zero_trades_",
)

metadata_cols = ("id", "date")

# Partition feature columns into K0 and K1 sets. Metadata columns pass through
# unchanged. Unmatched characteristics default to K0, which is the conservative
# assignment: it carries no incorrect temporal-structure assumption, though it
# may understate the lag value of some characteristics.
k1_cols = []
k0_cols = []
remaining = []

for col in feature_cols:
    if col in metadata_cols:
        continue
    if any(p in col for p in k1_factors):
        k1_cols.append(col)
    elif any(p in col for p in k0_factors):
        k0_cols.append(col)
    else:
        remaining.append(col)

k0_cols.extend(remaining)

print(f"K0(current only):{len(k0_cols):>4} characteristics")
print(f"K1(with annual lags):{len(k1_cols):>4} characteristics")
print(f"Unmatched(assigned K0):{len(remaining):>4}")

# Persist the K0 and K1 lists for the modelling notebook.
Path("..\\data\\k0_columns.json").write_text(pd.Series(k0_cols).to_json(orient="values"))
Path("..\\data\\k1_columns.json").write_text(pd.Series(k1_cols).to_json(orient="values"))

K0(current only): 308 characteristics
K1(with annual lags):  96 characteristics
Unmatched(assigned K0): 263


1030

### Missing Data Handling

In [ ]:
# Open the cleaned feature parquet without loading it. The dataset API allows
# us to filter by date and to read one month-end cross-section at a time
# through the underlying row-group structure

dataset = pa_dataset.dataset(FEATURES_RAW, format="parquet")
schema = dataset.schema
all_columns = schema.names

# Obtain the list of distinct dates via a single-column scan. PyArrow reads
# only the 'date' column from disk, which is cheap because parquet stores
# each column independently

print("Enumerating distinct month-end dates...")
date_table = dataset.to_table(columns=["date"])
all_dates = pd.to_datetime(pd.Series(date_table["date"].to_pylist())).unique()
all_dates = sorted(all_dates)
del date_table
gc.collect()
print(f"Distinct months: {len(all_dates):,}  ({all_dates[0].date()} to {all_dates[-1].date()})")

Enumerating distinct month-end dates...
Distinct months: 7,817  (2000-01-03 to 2025-12-31)


In [ ]:
# Identify characteristic columns once using a single month as a sample. The
# detection is dtype- and cardinality-based, so a single month-end cross-section
# is sufficient to classify every column consistently with the full panel

sample_date = all_dates[len(all_dates) // 2]   # mid-panel month for robustness
sample_filter = pa.compute.equal(pa.compute.field("date"), pa.scalar(sample_date))
sample_table = dataset.to_table(filter=sample_filter)
sample_df = sample_table.to_pandas()
del sample_table

print(f"Detecting characteristics on sample month {pd.Timestamp(sample_date).date()} "
      f"({len(sample_df):,} firms)...")

SAMPLE_SIZE = 50_000
char_cols = []

for col in sample_df.columns:
    if col in metadata_cols:
        continue
    series = sample_df[col]
    if pd.api.types.is_datetime64_any_dtype(series):
        continue
    if not pd.api.types.is_numeric_dtype(series):
        continue

    values = series.dropna()
    if values.empty:
        continue

    inner_sample = values.sample(SAMPLE_SIZE, random_state=0) if len(values) > SAMPLE_SIZE else values
    n_unique = inner_sample.nunique()
    unique_ratio = n_unique / max(len(inner_sample), 1)
    is_integer_like = bool(np.isclose(inner_sample, inner_sample.round()).all())
    is_binary = n_unique <= 2
    is_low_cardinality = n_unique < 50
    is_identifier_like = is_integer_like and unique_ratio > 0.80

    if is_binary or (is_integer_like and (is_low_cardinality or is_identifier_like)):
        continue

    char_cols.append(col)

print(f"Detected {len(char_cols)} characteristic columns out of {len(all_columns)} total columns.")
del sample_df
gc.collect()

Detecting characteristics on sample month 2013-06-07 (65 firms)...
Detected 370 characteristic columns out of 406 total columns.


31

In [ ]:

MISSING_THRESHOLD = 1/3

# Build the canonical output schema.
flag_fields_dict = {f"{c}_missing": c for c in char_cols}
output_fields = []
for field in schema:
    if field.name in flag_fields_dict.values():
        # Characteristic column: coerce to float64 to handle decimal/null types.
        output_fields.append(pa.field(field.name, pa.float64()))
    else:
        # Non-characteristic column: preserve the source type.
        output_fields.append(field)
for c in char_cols:
    output_fields.append(pa.field(f"{c}_missing", pa.int8()))
output_schema = pa.schema(output_fields)
print(f"Canonical output schema: {len(output_schema)} fields "
      f"({len(char_cols)} characteristics + {len(output_schema) - 2*len(char_cols)} other + "
      f"{len(char_cols)} missingness flags)")

# Accumulators for the aggregate missingness profile.
n_total_rows = 0
n_kept_rows = 0
miss_sum_per_char = np.zeros(len(char_cols), dtype=np.float64)
miss_count_per_char = np.zeros(len(char_cols), dtype=np.int64)
fraction_bins = np.zeros(6, dtype=np.int64)
bin_edges = [0.0, 0.10, 0.20, 1/3, 0.50, 0.75, 1.01]

# Create the writer eagerly with the canonical schema.
writer = pq.ParquetWriter(FEATURES_FLAG, output_schema, compression="snappy")

try:
    for i, t in enumerate(all_dates):
        # Read one month-end cross-section.
        month_filter = pc.equal(pc.field("date"), pa.scalar(t))
        month_table = dataset.to_table(filter=month_filter)
        df_month = month_table.to_pandas()
        del month_table

        if df_month.empty:
            continue

        # Coerce characteristic columns to float64 in pandas before further
        # processing. This guarantees consistent dtype regardless of whether
        # the source column was decimal128, all-null, or already float64.
        for col in char_cols:
            if col in df_month.columns:
                df_month[col] = pd.to_numeric(df_month[col], errors="coerce").astype(np.float64)

        # Accumulate aggregate statistics on the unfiltered cross-section.
        n_total_rows += len(df_month)
        char_block = df_month[char_cols]
        miss_per_row = char_block.isna().sum(axis=1).to_numpy(dtype=np.int32)
        miss_frac = miss_per_row / len(char_cols)
        miss_sum_per_char += char_block.isna().sum(axis=0).to_numpy(dtype=np.float64)
        miss_count_per_char += len(df_month)
        binned = np.clip(np.digitize(miss_frac, bin_edges, right=False) - 1, 0, 5)
        for b in range(6):
            fraction_bins[b] += int((binned == b).sum())

        # Apply the one-third missingness threshold.
        keep_mask = miss_frac <= MISSING_THRESHOLD
        df_month = df_month.loc[keep_mask].reset_index(drop=True)
        n_kept_rows += len(df_month)
        if df_month.empty:
            continue

        # Attach binary missingness indicator columns.
        flag_data = df_month[char_cols].isna().to_numpy(dtype=np.int8)
        flag_cols = [f"{c}_missing" for c in char_cols]
        flags_df = pd.DataFrame(flag_data, columns=flag_cols, index=df_month.index)
        df_month = pd.concat([df_month, flags_df], axis=1, copy=False)
        del flag_data, flags_df

        # Convert to Arrow and cast to the canonical schema before writing.
        # The cast handles any residual type drift (for example, the case
        # where a single month contains only NaN for one characteristic).
        out_table = pa.Table.from_pandas(df_month, preserve_index=False)
        out_table = out_table.cast(output_schema)
        writer.write_table(out_table)

        if (i + 1) % 50 == 0 or i + 1 == len(all_dates):
            print(f"Processed {i + 1:>4d} / {len(all_dates)} months  "
                  f"({100 * (i + 1) / len(all_dates):5.1f}%)  "
                  f"rows kept so far: {n_kept_rows:>10,}")

        del df_month, out_table
finally:
    writer.close()

print(f"\nStreaming pass complete.")
print(f"Rows before: {n_total_rows:>12,}")
print(f"Rows kept: {n_kept_rows:>12,}  ({n_kept_rows / n_total_rows:.1%})")
print(f"Rows dropped: {n_total_rows - n_kept_rows:>12,}  ({1 - n_kept_rows / n_total_rows:.1%})")

# Final aggregate missingness profile.
miss_by_char = pd.Series(miss_sum_per_char / miss_count_per_char, index=char_cols)
miss_by_char = miss_by_char.sort_values(ascending=False)

print("\nMost-missing characteristics")
print(miss_by_char.head(10).map("{:.1%}".format).to_string())
print(f"\nAverage characteristic missing rate: {miss_by_char.mean():.1%}")

bin_labels = ["0-10%", "10-20%", "20-33%", "33-50%", "50-75%", "75-100%"]
dist = pd.DataFrame({
    "rows": fraction_bins,
    "pct": [f"{c / n_total_rows:.1%}" for c in fraction_bins],
    "decision": ["keep", "keep", "keep", "drop", "drop", "drop"],
}, index=bin_labels)
print("\nPer-row missingness distribution")
print(dist.to_string())

print(f"\nFiltered panel with missingness flags saved to: {FEATURES_FLAG}")

gc.collect()

Canonical output schema: 776 fields (370 characteristics + 36 other + 370 missingness flags)


C:\Users\hemas\AppData\Local\Temp\ipykernel_21096\2010783137.py:86: Pandas4Warning: The copy keyword is deprecated and will be removed in a future version. Copy-on-Write is active in pandas since 3.0 which utilizes a lazy copy mechanism that defers copies until necessary. Use .copy() to make an eager copy if necessary.
  df_month  = pd.concat([df_month, flags_df], axis=1, copy=False)
C:\Users\hemas\AppData\Local\Temp\ipykernel_21096\2010783137.py:86: Pandas4Warning: The copy keyword is deprecated and will be removed in a future version. Copy-on-Write is active in pandas since 3.0 which utilizes a lazy copy mechanism that defers copies until necessary. Use .copy() to make an eager copy if necessary.
  df_month  = pd.concat([df_month, flags_df], axis=1, copy=False)
C:\Users\hemas\AppData\Local\Temp\ipykernel_21096\2010783137.py:86: Pandas4Warning: The copy keyword is deprecated and will be removed in a future version. Copy-on-Write is active in pandas since 3.0 which utilizes a lazy copy

Processed   50 / 7817 months  (  0.6%)  rows kept so far:     26,808


C:\Users\hemas\AppData\Local\Temp\ipykernel_21096\2010783137.py:86: Pandas4Warning: The copy keyword is deprecated and will be removed in a future version. Copy-on-Write is active in pandas since 3.0 which utilizes a lazy copy mechanism that defers copies until necessary. Use .copy() to make an eager copy if necessary.
  df_month  = pd.concat([df_month, flags_df], axis=1, copy=False)
C:\Users\hemas\AppData\Local\Temp\ipykernel_21096\2010783137.py:86: Pandas4Warning: The copy keyword is deprecated and will be removed in a future version. Copy-on-Write is active in pandas since 3.0 which utilizes a lazy copy mechanism that defers copies until necessary. Use .copy() to make an eager copy if necessary.
  df_month  = pd.concat([df_month, flags_df], axis=1, copy=False)
C:\Users\hemas\AppData\Local\Temp\ipykernel_21096\2010783137.py:86: Pandas4Warning: The copy keyword is deprecated and will be removed in a future version. Copy-on-Write is active in pandas since 3.0 which utilizes a lazy copy

KeyboardInterrupt: 